In [11]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import readsav
from pathlib import Path
import maos_utils as mu
from astropy.io import fits
import os

In [13]:
baseroot = Path("/u/bdigia/work/ao/keck/maos/keck/my_base")
seeds = np.array([1, 1000, 5000, 10000])

In [2]:
skyroot = Path("/Users/bdigia/work/ao/airopa_input/")
dates = [f.as_posix()[-16:-8] for f in skyroot.glob("*/")]
names = []
datecol = []
sky_paths = []
for date in dates:
    name = [f.as_posix()[-14:-9] for f in skyroot.glob(f"{date}nirc2_kp/*_psf.fits")]
    paths = [f.as_posix() for f in skyroot.glob(f"{date}nirc2_kp/*_psf.fits")]
    names.extend(name)
    temp = np.full(len(name), date)
    datecol.extend(temp.tolist())
    sky_paths.extend(paths)

names = np.array(names)
datecol = np.array(datecol)    
namesanddates = np.column_stack((names, datecol))

In [ ]:
telem_home = Path("/g/lu/data/keck_telemetry/")
_, telem_status = mu.find_on_sky_telemetry_file(datecol, 'LGS')
# On-sky nights for which telemetry exists
for i, sky in enumerate(namesanddates[telem_status == True]):
    sky_file = skyroot.as_posix() + f"/{sky[1]}nirc2_kp/{sky[0]}_psf.fits"
    sky_folder = skyroot.as_posix() + f"/{sky[1]}nirc2_kp/"

    # Find specific telemetry file for corresponding on-sky frame
    with fits.open(sky_paths[i]) as fits_file:
        hdu = fits_file[0]
        hdr = hdu.header
    try:
        prefix = hdr['FRAMENO']
        telem_path = telem_home.glob(f"{sky[1]}/sdata90*/nirc*/*/n{prefix}_LGS_trs.sav") 
    except KeyError:
        original = hdr['FILENAME']
        telem_path = telem_home.glob(f"{sky[1]}/sdata90*/nirc*/*/{original[:5]}_LGS_trs.sav") 
    
    # Load telemetry file
    try:
        data = readsav(telem_path)

        dt_gain = 0.0
        shwfs_gain = 0.0
        for item in data.header:
            decoded:str = item.decode('ascii')
            # TT loop gain 
            if decoded.startswith('DTGAIN'):
                gain = decoded.split(' ')[3]
                dt_gain = float(gain[1:])
            elif decoded.startswith('GAIN'):
                gain = decoded.split(' ')[2]
                shwfs_gain = float(gain[1:])

        # Zenith angle
        angle = np.degrees(np.arccos(1.0/float(hdr['AIRMASS'])))
        # Wavelength at which to run MAOS (microns)
        wvl = float(hdr['TARGWAVE']) * 1.0e-6 # multiply by 1e-6 to convert from microns to m
        # STRAP WFS integration time (milli-sec)
        if 'STINTTIM' not in hdr:
            continue
        hdr_strap_int_time = float(hdr['STINTTIM'])
        # SHWFS frame rate (Hz)
        hdr_shwfs_frame_rate = float(hdr['WSFRRT'])
        hdr_shwfs_int_time = (1.0/hdr_shwfs_frame_rate)*1000.0 # ms
        sim_dt = (1.0/472.0)*1000.0 # ms
        howfs_dtrat = int(sim_dt / hdr_shwfs_int_time)
        strap_dtrat = int(sim_dt / hdr_strap_int_time)
        # Calculate siglev/bkgrnd/nearecon config parameters using variable
        # integration times from headers. Will not use siglev and bkgrnd
        # since we are using telemetry data as inputs
        _, howfs_nearecon, _, howfs_bkgrnd = mu.keck_nea_photons(8.1, 'LGSWFS', hdr_shwfs_int_time/1000.0)
        _, strap_nearecon, _, _ = mu.keck_nea_photons(14.0, 'STRAP', hdr_strap_int_time/1000.0)
        # Compose powfs.dtrat array for input into MAOS config command override
        dtrat = [howfs_dtrat, strap_dtrat, 7080]
        # Telemetry APD_SKY_BACK, average 4 data points (one per quad) together for STRAP backgrnd,
        # multiplied by TT loop gain keyword
        strap_bkgrnd = np.mean(data.apd_sky_back[0])*dt_gain
        bkgrnd = [howfs_bkgrnd, strap_bkgrnd, 25.3]
        strap_siglev = np.mean(data.b.apdcounts[0])
        howfs_siglev = (np.mean(data.a.subapintensity[0]) * shwfs_gain) / hdr_shwfs_int_time
        siglev = [howfs_siglev, strap_siglev, 3723]
        nearecon = [howfs_nearecon, strap_nearecon, 8.4]
        # Calculate atm parameters
        fried, turbpro, windspds, winddrcts, _, _, _, _, _, _ = mu.estimate_on_sky_conditions(sky_file, 
                                                                                              sky_folder)
        # Size of on-sky image
        if int(hdr['NAXIS1']) == int(hdr['NAXIS2']):
            size = int(hdr['NAXIS1'])
        else:
            size = min(int(hdr['NAXIS1']), int(hdr['NAXIS2']))

        # MAOS seems to give weird warnings when input evl.psfsize parameter is odd
        if size % 2 != 0:
            size -= 1
        
        for simtype in ['piston', 'psd+ncpa-seen', 'psd+ncpa-unseen']:
            if simtype == 'piston':
                mode = 'piston'
                surf_cmd = ["Keck_ncpa_rmswfe130nm.fits"]
                psd_file = ''
            elif simtype == 'psd+ncpa-seen':
                mode = 'surf_wfs1'
                surf_cmd = ["Keck_ncpa_rmswfe130nm.fits", "'r0=0.36;l0=3.39;ht=40000;slope=-2; SURFWFS=1; SURFEVL=1; seed=10;'"]
                psd_file = "PSD_Keck_ws26.47mas_vib26mas_rad2.fits"
            elif simtype == 'psd+ncpa-unseen':
                mode = 'surf_wfs0'
                surf_cmd = ["Keck_ncpa_rmswfe130nm.fits", "'r0=0.36;l0=3.39;ht=40000;slope=-2; SURFWFS=0; SURFEVL=1; seed=10;'"]
                psd_file = "PSD_Keck_ws26.47mas_vib26mas_rad2.fits"
            else:
                raise ValueError(f"Invalid MAOS simulation type '{type}'. Valid types are currently: 'piston', 'psd+ncpa-seen', 'psd+ncpa-unseen'. See help() for further info")
    
            for seed in seeds:
                # Must be in MAOS simulation directory to run successfully
                if os.getcwd() != baseroot.as_posix():
                    os.chdir(baseroot)
    
                maos_cmd = f"maos -o A_keck_scao_lgs_gc_{mode}_comp_{sky[0]}_seed{seed}_epoch{sky[1]}_telemetry -c A_keck_scao_lgs_gc.conf evl.psfsize={size} sim.seeds={seed} evl.wvl={wvl} powfs.dtrat={dtrat} sim.zadeg={angle} powfs.siglev={siglev} powfs.bkgrnd={bkgrnd} powfs.nearecon={nearecon} sim.wspsd={psd_file} atm.r0z={fried} atm.wt={turbpro} atm.ws={windspds} atm.wddeg={winddrcts} surf={surf_cmd} -O"
                os.system(maos_cmd)
            
    except FileNotFoundError:
        # Do not run simulation if telemetry file doesn't exist
        continue

In [ ]:
    for i, sky in enumerate(framedates):
        sky_file = skyroot.as_posix() + f"/{sky[1]}nirc2_kp/{sky[0]}_psf.fits"
        sky_folder = skyroot.as_posix() + f"/{sky[1]}nirc2_kp/"
        
        with fits.open(sky_file) as fits_file:
            hdu = fits_file[0]
            hdr = hdu.header

        # Zenith angle
        angle = np.degrees(np.arccos(1.0/float(hdr['AIRMASS'])))
        # Wavelength at which to run MAOS (microns)
        wvl = float(hdr['TARGWAVE']) * 1.0e-6 # multiply by 1e-6 to convert from microns to m
        # STRAP WFS integration time (milli-sec)
        if 'STINTTIM' not in hdr:
            continue
        hdr_strap_int_time = float(hdr['STINTTIM'])
        # SHWFS frame rate (Hz)
        hdr_shwfs_frame_rate = float(hdr['WSFRRT'])
        hdr_shwfs_int_time = (1.0/hdr_shwfs_frame_rate)*1000.0 # ms
        sim_dt = (1.0/472.0)*1000.0 # ms
        howfs_dtrat = int(sim_dt / hdr_shwfs_int_time)
        strap_dtrat = int(sim_dt / hdr_strap_int_time)
        # Compose powfs.dtrat array for input into MAOS config command override
        dtrat = [howfs_dtrat, strap_dtrat, 7080]
        # Calculate siglev/bkgrnd/nearecon config parameters using variable
        # integration times from headers
        _, howfs_nearecon, howfs_siglev, howfs_bkgrnd = keck_nea_photons(8.1, 'LGSWFS', 
                                                                         hdr_shwfs_int_time/1000.0)
        _, strap_nearecon, strap_siglev, strap_bkgrnd = keck_nea_photons(14.0, 'STRAP', 
                                                                         hdr_strap_int_time/1000.0)
        nearecon = [howfs_nearecon, strap_nearecon, 8.4]
        siglev = [howfs_siglev, strap_siglev, 3723]
        bkgrnd = [howfs_bkgrnd, strap_bkgrnd, 25.3]
        # Calculate atm parameters
        fried, turbpro, windspds, winddrcts, _, _, _, _, _, _ = estimate_on_sky_conditions(sky_file, 
                                                                                           sky_folder)
        # Size of on-sky image
        if int(hdr['NAXIS1']) == int(hdr['NAXIS2']):
            size = int(hdr['NAXIS1'])
        else:
            size = min(int(hdr['NAXIS1']), int(hdr['NAXIS2']))

        # MAOS seems to give weird warnings when input evl.psfsize parameter is odd
        if size % 2 != 0:
            size -= 1
        
        mode = ''
        if simtype == 'piston':
            mode = 'piston'
            surf_cmd = ["Keck_ncpa_rmswfe130nm.fits"]
            # Fetch name of current input PSD FITS file in MAOS config file keck_sim.conf
            psd_file = ''
        elif simtype == 'psd+ncpa-seen':
            mode = 'surf_wfs1'
            surf_cmd = ["Keck_ncpa_rmswfe130nm.fits", "'r0=0.36;l0=3.39;ht=40000;slope=-2; SURFWFS=1; SURFEVL=1; seed=10;'"]
            psd_file = "PSD_Keck_ws26.47mas_vib26mas_rad2.fits"
        elif simtype == 'psd+ncpa-unseen':
            mode = 'surf_wfs0'
            surf_cmd = ["Keck_ncpa_rmswfe130nm.fits", "'r0=0.36;l0=3.39;ht=40000;slope=-2; SURFWFS=0; SURFEVL=1; seed=10;'"]
            psd_file = "PSD_Keck_ws26.47mas_vib26mas_rad2.fits"
        else:
            raise ValueError(f"Invalid MAOS simulation type '{type}'. Valid types are currently: 'piston', 'psd+ncpa-seen', 'psd+ncpa-unseen'. See help() for further info")

        for seed in seeds:
            # Must be in MAOS simulation directory to run successfully
            if os.getcwd() != baseroot.as_posix():
                os.chdir(baseroot)

            maos_cmd = f"maos -o A_keck_scao_lgs_gc_{mode}_comp_{sky[0]}_seed{seed}_epoch{sky[1]} -c A_keck_scao_lgs_gc.conf evl.psfsize={size} sim.seeds={seed} evl.wvl={wvl} powfs.dtrat={dtrat} sim.zadeg={angle} powfs.siglev={siglev} powfs.bkgrnd={bkgrnd} powfs.nearecon={nearecon} sim.wspsd={psd_file} atm.r0z={fried} atm.wt={turbpro} atm.ws={windspds} atm.wddeg={winddrcts} surf={surf_cmd} -O"
            os.system(maos_cmd)